In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Attention Scoring and Masking

In that section the attention weights came from a kernel
chosen by hand. They follow distance in a fixed input space, and no amount
of data changes what counts as similar. In a sentence, however, a verb looking for
its subject wants the word that plays that grammatical role, which need not
be the word whose representation is closest to its own. Turning soft lookup
into a layer of a trained network raises two questions: which score should
the network learn, and which query--key pairs should count at all?

The score must be shaped by learned representations, and it must be cheap,
because it is evaluated for every pair: one attention head of GPT-2 small,
reading a context of 1,024 tokens, scores $1{,}024^2 = 1{,}048{,}576$
pairs. Some of those pairs must not count. Sequences of different lengths
share a minibatch only after padding, and no query should attend to the
padding. A language model trained to predict each next token processes all
positions at once, and the query at position $t$ must not see the tokens
after it.

This section answers both questions. The score is the dot product of query and
key, divided by $\sqrt{d}$ so that its scale does not grow with the width
$d$ of the vectors. Exclusions are boolean masks applied before the
softmax, and several of them combine by logical AND. With both in place,
attention over a whole minibatch is two batched matrix multiplications and
a softmax, the `DotProductAttention` layer that every later model in this
book builds on. The section ends with the translation model in which
learned attention first appeared.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Dot-Product Attention

We look for a score that learned queries and keys can shape and that costs
one short computation per pair. The Gaussian kernel of
that section already contains one. Expanding its
exponent gives the score

$$
a(\mathbf{q}, \mathbf{k}_i) = -\frac{1}{2} \|\mathbf{q} - \mathbf{k}_i\|^2  = \mathbf{q}^\top \mathbf{k}_i -\frac{1}{2} \|\mathbf{k}_i\|^2  -\frac{1}{2} \|\mathbf{q}\|^2.
$$

The last term depends on the query only. It is the same for all keys, and
the softmax normalization the equation removes it. If all
keys have the same norm, the middle term drops out in the same way, and
the Gaussian kernel and the dot product give identical weights. In general
the norms differ, and keeping only $\mathbf{q}^\top \mathbf{k}_i$ is a
modeling decision rather than an approximation: the dot product becomes
the compatibility function in its own right. Learned query and key
representations then decide what compatibility means, and the Gaussian
kernel remains the special case of keys with equal norms.

One adjustment controls the size of the scores. Suppose that the $d$
coordinates of the query and of the key are independent random variables
with mean zero and variance one. Their dot product then has mean zero and
variance $d$; for the GPT-2 head, with $d = 64$, a typical score is about
$\pm 8$. Dividing by $\sqrt{d}$ restores unit variance and gives the
*scaled dot-product* score of the Transformer
[@Vaswani.Shazeer.Parmar.ea.2017]:

$$ a(\mathbf{q}, \mathbf{k}_i) = \mathbf{q}^\top \mathbf{k}_i / \sqrt{d}.$$

The attention weights are, as always, the softmax of the scores:

$$\alpha(\mathbf{q}, \mathbf{k}_i) = \mathrm{softmax}(a(\mathbf{q}, \mathbf{k}_i)) = \frac{\exp(\mathbf{q}^\top \mathbf{k}_i / \sqrt{d})}{\sum_{j=1}^m \exp(\mathbf{q}^\top \mathbf{k}_j / \sqrt{d})}.$$

### Score Variance and the $1/\sqrt{d}$ Factor

Learned queries and keys are neither independent nor of unit variance, so
the calculation above describes a network at initialization. That is when
it matters. If one score exceeds the others by a wide margin, its weight
approaches $1$, the others approach $0$, and the softmax *saturates*. The
gradient that reaches the scores shrinks with it, because the Jacobian of
the softmax is

$$
\frac{\partial \boldsymbol{\alpha}}{\partial \mathbf{a}} = \mathrm{diag}(\boldsymbol{\alpha}) - \boldsymbol{\alpha} \boldsymbol{\alpha}^\top,
$$

which tends to the zero matrix as $\boldsymbol{\alpha}$ approaches a
one-hot vector. Row $i$ of the Jacobian carries the factor $\alpha_i$, so
the gradient of the loss with respect to score $i$ is proportional to the
weight of key $i$. In a saturated row, the keys with negligible weight
receive almost no gradient, and the query learns only about the few keys
that already carry weight. A key that loses at the start has little
chance to catch up. Only this route saturates. Gradients still reach the
values, the output projection, and any residual connection. For finite
scores the Jacobian is never exactly zero, since the all-ones vector alone
spans its null space (an exercise below proves this), but it can come
arbitrarily close.

We measure both effects. We draw random queries and keys with independent
standard normal entries, compute the attention weights of each query over
$64$ keys, and record the entropy of the weights as the dimension $d$
grows, with and without the factor $1/\sqrt{d}$. For reference, the
uniform distribution over $64$ keys has entropy $\log 64 \approx 4.2$
nats, and a one-hot distribution has entropy $0$.

In [ ]:
def softmax_weights(d, scale, num_keys=64, num_queries=256):
    """Log-weights of random queries over random keys, scores scaled."""
    q = torch.randn(num_queries, d)
    keys = torch.randn(num_queries, num_keys, d)
    scores = (keys @ q[..., None]).squeeze(-1) * scale
    return F.log_softmax(scores, dim=-1)

def entropy(log_alpha):
    return -(log_alpha.exp() * log_alpha).sum(-1).mean().item()

def jacobian_norm(log_alpha):
    """Mean Frobenius norm of the Jacobian diag(alpha) - alpha alpha^T."""
    alpha = log_alpha.exp()
    J = torch.diag_embed(alpha) - alpha[..., :, None] * alpha[..., None, :]
    return torch.linalg.matrix_norm(J).mean().item()

The entropy uses the log-weights directly, so a weight that underflows to
zero contributes exactly zero rather than $0 \cdot \log 0$.

In [ ]:
ds = [2**k for k in range(2, 10)]
entropies = [[entropy(softmax_weights(d, 1.0)) for d in ds],
             [entropy(softmax_weights(d, 1 / math.sqrt(d))) for d in ds]]
d2l.plot(ds, entropies, 'dimension d', 'entropy (nats)',
         legend=['unscaled', 'scaled by 1/sqrt(d)'], xscale='log')

With scaling, the entropy stays near $3.7$ nats over two orders of
magnitude of $d$: the weights are equally sharp at every width. Without
scaling it collapses as $d$ grows, to about $0.2$ nats at $d = 512$, a
nearly one-hot weighting that comes from the length of the vectors alone.
Unscaled, the largest of the $64$ weights takes about $80\%$ of the total
at $d = 64$ and about $90\%$ at $d = 512$. The norm of the Jacobian shows
what this does to the gradient that reaches the scores:

In [ ]:
for d in (4, 64, 512):
    unscaled = jacobian_norm(softmax_weights(d, 1.0))
    scaled = jacobian_norm(softmax_weights(d, 1 / math.sqrt(d)))
    print(f'd = {d:3d}: Jacobian norm {unscaled:.3f} (unscaled), '
          f'{scaled:.3f} (scaled)')

With scaling, the norm is the same at every dimension. Without it, the
norm first *rises*: at small $d$ the unscaled weights concentrate on a few
keys, and the Jacobian is largest when two keys share the weight (an
exercise below finds this maximum). It crosses the scaled value near
$d = 128$, falls to about half of it at $d = 512$, and keeps falling. A
model with unscaled scores would therefore start training with nearly
one-hot attention, and the gradient that reaches its scores would flow
almost entirely through the few keys that each query already favors. The
factor $1/\sqrt{d}$ removes this dependence on the dimension. That is why
it is part of the definition the equation rather
than a hyperparameter to tune.

## Masking

So far every query attends to every key. Two situations require
exclusions. The first is *padding*. Sequences of different lengths share
a minibatch only after they are padded with a dummy token to a common
length (here `<blank>`):

```
Dive  into  Deep    Learning
Learn to    code    <blank>
Hello world <blank> <blank>
```

The padding carries no meaning, and no query should give it weight. The
second is *causality*. A language model trained to predict the next token
computes its outputs for all positions of a sequence at once, and the
query at position $t$ must not see the keys at positions after $t$.
Otherwise it could copy the very token it is asked to predict.

Both are described by one object, a boolean *mask* with one entry per
query--key pair that is true where the query may attend to the key. The
true entries of a row need not form a prefix, so the same object also
describes the attention windows and document boundaries we meet later.

### The Masked Softmax Operation

The *masked softmax* gives zero weight to the excluded pairs. It does not
skip them, which would leave rows of different lengths. Instead it
overwrites the excluded scores with a very negative number before the
softmax, so that their exponentials vanish, and keeps the regular array
shapes that batched kernels need.

The number has to be chosen with the precision in mind. Older code used a
constant such as $-10^6$. That works in single precision. Float16, a format
used for training today, cannot represent numbers beyond about
$6.5 \times 10^4$, and depending on the framework $-10^6$ either
overflows to $-\infty$ or is refused with an error. Literal $-\infty$
also masks correctly, until some query has no valid key at all. Every
score of its row is then $-\infty$, the softmax subtracts the row maximum
and computes $-\infty - (-\infty)$, and the row becomes NaN. The NaN
spreads to everything computed from it. We therefore overwrite with the most negative
*finite* value of the dtype (`finfo(dtype).min`). After the softmax
subtracts the row maximum, its exponential underflows to exactly zero at
any precision, and no NaN can arise.

One case remains. A query whose keys are all excluded would now receive
the uniform average of invalid values, because all its scores are equal.
`masked_softmax` sets such rows to zero. A model should still ensure that
every query keeps at least one valid key.

In [ ]:
def masked_softmax(X, mask=None):
    """Softmax over the last axis with zero weight where mask is False."""
    if mask is None:
        return F.softmax(X, dim=-1)
    # Most negative finite value: zero weight at any precision, no NaN
    X = X.masked_fill(~mask, torch.finfo(X.dtype).min)
    weights = F.softmax(X, dim=-1)
    # A query without any valid key gets zero weights, not a uniform row
    return torch.where(mask.any(-1, keepdim=True), weights, 0.0)

The mask need only *broadcast* against the scores, so a mask that is the
same for every query, or for every sequence, is stored once. A padding
mask is of this kind: key $j$ of a sequence is valid when $j$ is less than
the sequence's valid length. `padding_mask` turns a vector of valid
lengths, one per sequence, into a mask of shape (batch, 1, keys), which
the softmax broadcasts over the queries. A matrix of lengths, one per
query, gives a mask of shape (batch, queries, keys).

In [ ]:
def padding_mask(valid_lens, num_keys):
    """Boolean mask that is True for the first valid_lens keys."""
    # valid_lens of shape (batch,) gives (batch, 1, keys); per-query
    # lengths of shape (batch, queries) give (batch, queries, keys)
    if valid_lens.dim() == 1:
        valid_lens = valid_lens[:, None]
    keys = torch.arange(num_keys, device=valid_lens.device)
    return keys < valid_lens[..., None]

We try both on a minibatch of two sequences with two queries and four
keys each, and with valid lengths $2$ and $3$. The weights beyond the
valid length should be zero, and each row should still sum to $1$.

In [ ]:
masked_softmax(torch.rand(2, 2, 4), padding_mask(torch.tensor([2, 3]), 4))

With one valid length per query, each row is cut at its own length:

In [ ]:
masked_softmax(torch.rand(2, 2, 4),
               padding_mask(torch.tensor([[1, 3], [2, 4]]), 4))

### Causal Masking

In a causal mask, the query at position $i$ may see the keys at positions
$j \leq i$. The mask is a lower triangle, and in a sequence of length $n$
it excludes $n(n-1)/2$ of the $n^2$ pairs, almost half of them. For one head
of GPT-2 small reading $n = 1{,}024$ tokens, that is $523{,}776$ of the $1{,}048{,}576$
scores.
`causal_mask` also covers the case of fewer queries than keys, which
arises when new tokens attend to the keys of earlier tokens kept in a
cache (that section). The queries are then the *last* positions
of the sequence, and query $i$ may see key $j$ when $j \leq i +
(\textrm{keys} - \textrm{queries})$.

In [ ]:
def causal_mask(num_queries, num_keys=None, device=None):
    """Boolean (queries, keys) mask, True where a query may see a key."""
    # Queries are the last positions, as when new tokens attend to a
    # cache: query i sees key j if j <= i + num_keys - num_queries
    num_keys = num_queries if num_keys is None else num_keys
    i = torch.arange(num_queries, device=device)[:, None]
    j = torch.arange(num_keys, device=device)[None, :]
    return j <= i + (num_keys - num_queries)

On six random scores, the masked weights form a lower triangle:

In [ ]:
scores = torch.randn(1, 6, 6)
d2l.show_heatmaps(masked_softmax(scores, causal_mask(6))[None],
                  xlabel='Keys', ylabel='Queries')

The first query attends only to itself, and each later query spreads its
weight over itself and its predecessors. The triangle lets a model train
on all positions of a sequence in parallel without showing any position
its future. Every decoder in the following chapters uses it.

### Composing Masks

Each requirement excludes some pairs. Padding excludes the keys beyond a
sequence's length, causality the keys after the query, and the attention
windows of that section the keys too far from it. A
pair is allowed only if every requirement allows it, so masks compose by
logical AND. Broadcasting keeps the composite small. A padding mask of
shape (batch, 1, keys) and a causal mask of shape (queries, keys) combine
into a mask of shape (batch, queries, keys), and neither is copied per
example or per query before the AND (the figure).

![Masks compose by logical AND. For a sequence of length 6 with valid length 3, the padding mask stores one row of keys, which broadcasting repeats for every query (faint); the causal mask is a lower triangle; their AND allows a pair only if both do. Rows are queries, columns are keys, and filled entries are allowed.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-mask-algebra.svg)

Here are the weights for a minibatch of two sequences of length $6$ with
valid lengths $6$ and $3$:

In [ ]:
n = 6
padding = padding_mask(torch.tensor([6, 3]), n)  # (batch, 1, key)
causal = causal_mask(n)                          # (query, key)
mask = padding & causal                          # (batch, query, key)
weights = masked_softmax(torch.randn(2, n, n), mask)
d2l.show_heatmaps(weights[None], xlabel='Keys', ylabel='Queries',
                  titles=['valid length 6', 'valid length 3'],
                  figsize=(5, 2.5))

The first sequence shows the plain causal triangle. The second has valid
length $3$, so every row stops at the third key. Composition makes empty
rows more likely. Masks that each leave every query a key can still have
an empty intersection, so the guarantee of a valid key must hold for the
composite mask.

The same algebra handles *packed sequences*. Padding many short documents
to a common length wastes computation on the padding, so training
pipelines concatenate documents into one row instead. A mask that allows a
pair only within one document, ANDed with the causal mask, keeps the
documents apart. Here a row of length $6$ holds three documents of
lengths $3$, $2$, and $1$:

In [ ]:
doc = torch.tensor([0, 0, 0, 1, 1, 2])  # Document of each position
same_doc = doc[:, None] == doc[None, :]
packed = same_doc & causal_mask(6)
d2l.show_heatmaps(packed.float()[None, None], xlabel='Keys',
                  ylabel='Queries', cbar=False)

The composite mask is block diagonal, with one causal triangle per
document: each position attends only to the earlier positions of its own
document.

## Batched Attention

### Batch Matrix Multiplication

Attention is computed on minibatches of queries, keys, and values, so we
need to multiply batches of matrices with each other. Consider two
batches of $n$ matrices each,

$$
\mathbf{Q} = [\mathbf{Q}_1, \mathbf{Q}_2, \ldots, \mathbf{Q}_n]  \in \mathbb{R}^{n \times a \times b}, \qquad
\mathbf{K} = [\mathbf{K}_1, \mathbf{K}_2, \ldots, \mathbf{K}_n]  \in \mathbb{R}^{n \times b \times c}.
$$

The *batch matrix multiplication* (BMM) computes one matrix product per
batch element,

$$\textrm{BMM}(\mathbf{Q}, \mathbf{K}) = [\mathbf{Q}_1 \mathbf{K}_1, \mathbf{Q}_2 \mathbf{K}_2, \ldots, \mathbf{Q}_n \mathbf{K}_n] \in \mathbb{R}^{n \times a \times c}.$$

The `@` operator computes it. It batches over any number of leading
axes, which the attention layers of the next sections use for their extra
axis of heads.

In [ ]:
Q = torch.ones((2, 3, 4))
K = torch.ones((2, 4, 6))
d2l.check_shape(Q @ K, (2, 3, 6))

`torch.bmm(Q, K)` computes the same product for arrays with exactly one
batch axis. The `@` operator, short for `torch.matmul`, also accepts any
number of leading axes and broadcasts them.

### The DotProductAttention Class

Scaled dot-product attention can now be written for many queries at once.
For $n$ queries and $m$ key--value pairs, with queries and keys of width
$d$ and values of width $d_v$, stack the queries into $\mathbf{Q} \in
\mathbb{R}^{n \times d}$, the keys into $\mathbf{K} \in \mathbb{R}^{m
\times d}$, and the values into $\mathbf{V} \in \mathbb{R}^{m \times
d_v}$. The whole computation is two matrix products and a softmax
(the figure):

$$ \mathrm{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^\top }{\sqrt{d}}\right) \mathbf{V} \in \mathbb{R}^{n \times d_v}.$$

![Shapes in scaled dot-product attention. $\mathbf{Q}\mathbf{K}^\top/\sqrt{d}$ holds the scores of all $n \times m$ query--key pairs; masking and a softmax over each row turn them into the weights $\mathbf{A}$ (here with a causal mask, which leaves the upper triangle empty); multiplying by $\mathbf{V}$ gives one weighted sum of values per query.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-shapes.svg)

The softmax normalizes each row of the score matrix, so row $i$ of the
output is the attention the equation of query $i$. The
cost is visible in the shapes. For the GPT-2 head at a context of $n = m
= 1{,}024$ with $d = d_v = 64$, the score matrix holds $1{,}048{,}576$
entries, 4 MiB in single precision, and the two products take $2 \cdot 2
n^2 d \approx 0.27$ billion floating-point operations. Both numbers grow
with the square of the context length, which is the subject of
that section.

Queries and keys must have the same width $d$. That is no restriction in
practice, because the queries and keys of a network are computed from its
inputs by learned linear maps, which can map to any common width
(that section). On a minibatch,
the equation uses the batch matrix multiplication
the equation twice. The class applies dropout to the
attention weights as a regularizer and keeps the weights of its last call
in `attention_weights` for inspection.

In [ ]:
class DotProductAttention(nn.Module):
    """Scaled dot-product attention over the last two axes."""
    def __init__(self, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.attention_weights = None

    # Leading axes (batch, and heads if present) are carried along.
    # Shape of queries: (..., no. of queries, d)
    # Shape of keys: (..., no. of key-value pairs, d)
    # Shape of values: (..., no. of key-value pairs, value dimension)
    # mask: boolean, broadcastable to (..., no. of queries, no. of keys),
    # True where a query may attend to a key
    def forward(self, queries, keys, values, mask=None):
        d = queries.shape[-1]
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, mask)
        return self.dropout(self.attention_weights) @ values

We test the class on a minibatch of two sequences, each with one query of
width $2$ and ten key--value pairs with values of width $4$, and with
valid lengths $2$ and $6$. Evaluation mode switches dropout off. The
output has one row of width $4$ per query.

In [ ]:
queries = torch.randn(2, 1, 2)
keys = torch.randn(2, 10, 2)
values = torch.randn(2, 10, 4)
mask = padding_mask(torch.tensor([2, 6]), 10)

attention = DotProductAttention(dropout=0.5)
attention.eval()
d2l.check_shape(attention(queries, keys, values, mask), (2, 1, 4))

The stored weights confirm the mask. Each row holds the weights of one
sequence's query, and they vanish beyond the second and the sixth key,
respectively.

In [ ]:
d2l.show_heatmaps(attention.attention_weights.reshape((1, 1, 2, 10)),
                  xlabel='Keys', ylabel='Sequences', figsize=(4, 1.6))

## Attention as Learned Alignment

Learned attention first appeared in neural machine translation. Around
2014 the leading neural approach encoded a source sentence with an RNN
into one fixed-size vector and decoded the translation from that vector
with a second RNN; we build such an encoder--decoder in
that section. This is the bottleneck of
that section: one vector had to represent the whole
sentence, and translation quality fell as sentences grew longer.

@Bahdanau.Cho.Bengio.2014 kept the two RNNs but changed what the
decoder reads. At every output step, the current decoder state served as a
query against all encoder states, which served as keys and values. The
weighted sum, a fresh one for every output token, entered the next
prediction. The paper's title called this "jointly learning to align and
translate". The learned weights resembled the soft word alignments of
classical statistical translation: mostly monotone along the diagonal,
with departures where the two languages order words differently, as
sketched in the figure. Nothing in the training objective
asked for alignments. They emerged because they help to translate.

![Soft alignment between an English sentence and its French translation, in the style of a learned attention map (darker cells indicate larger weight; schematic). The alignment is mostly monotone, but "black cat" maps to "chat noir" with the order reversed, and both "était" and "assis" draw on "sat".](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-attention-alignment.svg)

One detail differed from the score we chose above. The decoder state and
the encoder states had different widths, so instead of a dot product,
@Bahdanau.Cho.Bengio.2014 scored with a small network with one
hidden layer, now called *additive attention*:

$$a(\mathbf{q}, \mathbf{k}) = \mathbf{w}_v^\top \tanh(\mathbf{W}_q \mathbf{q} + \mathbf{W}_k \mathbf{k}) \in \mathbb{R}.$$

Here $\mathbf{q} \in \mathbb{R}^{d_q}$ and $\mathbf{k} \in
\mathbb{R}^{d_k}$ may have different widths, and the learned parameters
are $\mathbf{W}_q \in \mathbb{R}^{r \times d_q}$, $\mathbf{W}_k \in
\mathbb{R}^{r \times d_k}$, and $\mathbf{w}_v \in \mathbb{R}^{r}$ for a
hidden width $r$. The two projections map queries and keys into a shared
$r$-dimensional space, and $\mathbf{w}_v$ reads a score off their sum. We
score three queries of width $20$ against six keys of width $2$, widths
that no dot product could pair.

In [ ]:
queries, keys = torch.randn(3, 20), torch.randn(6, 2)
r = 8  # Hidden width
W_q = torch.randn(r, 20) / math.sqrt(20)
W_k = torch.randn(r, 2) / math.sqrt(2)
w_v = torch.randn(r) / math.sqrt(r)
features = torch.tanh((queries @ W_q.T)[:, None, :]
                      + (keys @ W_k.T)[None, :, :])
scores = features @ w_v
d2l.check_shape(scores, (3, 6))
F.softmax(scores, dim=-1)

Each row is a distribution over the six keys. The intermediate array
`features` has shape $3 \times 6 \times 8$, one hidden vector per
query--key pair, and this array is why the dot product replaced the
additive score. All dot products of queries and keys form one matrix
multiplication, which accelerators execute fast, while the additive score
needs an $n \times m \times r$ array of hidden activations. The dot
product can still use a learned metric. Projecting queries and keys with
learned matrices first leaves a score that runs at the speed of matrix
multiplication. This is the form attention takes in the Transformer,
whose authors dropped the recurrent networks altogether and kept attention
as the only operation that relates positions
[@Vaswani.Shazeer.Parmar.ea.2017].

We can now compute attention for a whole minibatch, with scores that
learned representations can shape and with any pattern of exclusions.
Each query, however, still forms a single weighted average, and nothing in
the computation knows where in the sequence a key sits.
that section runs several lookups side by side, and
that section adds positions.

## Summary

Scaled dot-product attention scores a query against a key by
$\mathbf{q}^\top \mathbf{k}/\sqrt{d}$, which is the Gaussian kernel without
the norm terms, and it scores all pairs of a minibatch with one batched
matrix multiplication. The factor $1/\sqrt{d}$ keeps the score variance
independent of the width, so the softmax does not saturate at
initialization merely because the vectors are long. A boolean mask
excludes padding, future positions, or any other pairs before the
softmax. Masks compose by logical AND under broadcasting. The excluded
scores are overwritten with the most negative finite value of the dtype,
which is safe in reduced precision. `DotProductAttention` combines
scoring, masking, dropout, and the weighted sum of values. Additive
attention, the score of the first neural alignment model, gave way to the
dot product because the dot product is a matrix multiplication.

## Exercises

1. **Invariances of the score.** The Gaussian score
   the equation and the scaled dot product
   the equation both define attention weights through
   the softmax the equation.
    1. For each of the two scores, determine which of the following
       transformations leaves the weights unchanged: adding one vector
       $\mathbf{c}$ to every key; adding $\mathbf{c}$ to the query and to
       every key; multiplying the query and every key by one orthogonal
       matrix.
    1. Queries and keys are usually computed from inputs by affine maps,
       $\mathbf{q} = \mathbf{W}_q \mathbf{x} + \mathbf{b}_q$ and
       $\mathbf{k}_i = \mathbf{W}_k \mathbf{y}_i + \mathbf{b}_k$. Expand
       $\mathbf{q}^\top \mathbf{k}_i$ into four terms and show that the
       weights do not depend on $\mathbf{b}_k$. What is the gradient of the
       loss with respect to $\mathbf{b}_k$?
    1. Through which of the four terms does $\mathbf{b}_q$ affect the
       weights, and what preference over the keys can it express that the
       bias-free score cannot?

1. [code] **The softmax Jacobian.** `jacobian_norm` measures the
   Frobenius norm of the Jacobian the equation.
    1. Derive the equation from
       the equation, show that for finite scores its null
       space is spanned by the all-ones vector, and check the formula
       against automatic differentiation on a random score vector.
    1. Compute the Frobenius norm when $\boldsymbol{\alpha}$ is one-hot and
       when it is uniform over $m$ keys, and evaluate the uniform case for
       $m = 64$. Show that the squared norm is close to $\sum_i \alpha_i^2$
       when every weight is small, and use this to explain why the scaled
       norms printed for $d \in \{4, 64, 512\}$ exceed the uniform value.
    1. Compute the norm for weights spread uniformly over $k$ of the $m$
       keys, find the $k$ that maximizes it, and confirm by sampling random
       weight vectors that none exceeds this maximum. Use the result to
       explain why the unscaled norm printed for $d = 4$ exceeds the scaled
       one.

1. **Fully masked rows.** `masked_softmax` overwrites excluded scores with
   the most negative finite value of the dtype and then zeroes the rows of
   queries that have no valid key.
    1. Explain why, without the final zeroing, a fully masked row returns
       the uniform distribution, and why the same row returns NaN when the
       scores are overwritten with $-\infty$ instead. Trace both cases
       through the subtraction of the row maximum that a numerically stable
       softmax performs.
    1. In a self-attention layer, whose queries, keys, and values all come
       from the same sequence, let a query at a padding position have no
       valid key, as the composite masks of that section
       can arrange, and let the excluded scores be overwritten with
       $-\infty$, so that the output at that position is NaN. That output
       is discarded, and in the next attention layer every valid query
       excludes the padding position as a key. Explain why the NaN
       nevertheless corrupts the outputs of all valid positions of that
       sequence in the next layer.

1. **Composite masks.** that section combines masks by
   logical AND and warns that masks that are harmless alone can leave a
   query with no valid key. A causal sliding window of width $w$ lets the
   query at position $t$ attend to the keys at positions $t - w + 1, \ldots,
   t$.
    1. For a sequence of length $6$ with valid length $3$, write the padding
       mask and the causal sliding window of width $2$ as $6 \times 6$
       boolean matrices. Check that each leaves every query at least one
       key, and find the queries whose row of the AND is empty.
    1. Show that an AND of masks has no empty row if every mask allows each
       query to attend to its own position. Which of the padding, causal,
       sliding-window, and packed-sequence masks satisfy this condition, and
       what does the answer say about the empty rows you found?
    1. Some implementations add a bias matrix with entries $0$ for allowed
       pairs and $c < 0$ for excluded pairs, and compose masks by adding
       their biases. Take for $c$ the most negative finite single-precision
       value, about $-3.4 \times 10^{38}$. Compute the sum of two such
       biases in single precision, and determine what the softmax returns
       for a row in which every key is excluded by both masks and for a row
       in which every key is excluded but some keys by only one mask.

1. [code] **Attention dropout.** `DotProductAttention` applies dropout with
   rate $p$ to the attention weights, and the weights it exposes for
   visualization are those before dropout. Dropout multiplies each kept
   weight by $1/(1 - p)$ and sets the others to zero, so a row of weights
   no longer sums to $1$.
    1. Show that the row sum after dropout has expectation $1$ and variance
       $\frac{p}{1-p} \sum_i \alpha_i^2$, and evaluate the variance for a
       row that is uniform over $l$ keys and for a one-hot row. What is the
       output of a one-hot row in the draws in which dropout removes the
       attended weight?
    1. Use the inputs of that section with
       dropout $0.5$ active, and pass as values a batch of $10 \times 10$
       identity matrices, so that each output row equals the weights after
       dropout. Compare the mean and the variance of the row sums over many
       draws with the prediction computed from the weights before dropout.

1. **Entropy of random scores.** ● The scaled entropy curve above
   stays near $3.7$ nats, below the uniform value $\log 64 \approx 4.16$.
   Fix one query $\mathbf{q} \in \mathbb{R}^d$ and draw $m$ keys with
   independent standard normal coordinates, as `softmax_weights` does.
    1. Show that the scaled scores $a_i = \mathbf{q}^\top \mathbf{k}_i /
       \sqrt{d}$ are independent normal variables with mean $0$ and variance
       $s^2 = \|\mathbf{q}\|^2 / d$, and compute the expected value of $s^2$
       when $\mathbf{q}$ is drawn as in `softmax_weights`.
    1. The entropy of the weights is $H = \log Z - \sum_i \alpha_i a_i$ with
       $Z = \sum_j \exp(a_j)$. For large $m$, replace $Z/m$ and
       $\sum_i \exp(a_i)\, a_i / m$ by their expected values and derive
       $$H \approx \log m - \frac{s^2}{2}.$$
       Evaluate the approximation for $m = 64$ and $s = 1$.
    1. The approximation needs $Z$ to concentrate around its mean. Compute
       the relative standard deviation of $Z$ as a function of $s$ and $m$,
       and derive from it a condition on $s^2$ under which the approximation
       holds. Unscaled scores have $s^2 = \|\mathbf{q}\|^2$, whose expected
       value is $d$. Evaluate the relative standard deviation for $m = 64$
       at $s^2 = 4$ and $s^2 = 8$, and compare the approximation with the
       unscaled entropies plotted above for $d = 4$ and
       $d = 8$.

1. [code] **Additive scores near zero.** ● The additive score
   the equation adds the projected query and the projected key
   inside a $\tanh$.
    1. Replace $\tanh$ by the identity and show that the attention weights
       then do not depend on the query.
    1. Expand $\tanh$ to third order around zero, identify the lowest-order
       terms through which the weights depend on the query, and state how
       this dependence scales when $\mathbf{W}_q$ and $\mathbf{W}_k$ are
       both multiplied by a small factor $\epsilon$.
    1. Multiply `W_q` and `W_k` in the code of
       that section by $\epsilon \in \{1, 0.3, 0.1,
       0.03\}$, and measure the largest difference between the weight rows
       of the three queries. Check the scaling you derived, and state what
       the result for $\epsilon = 1$ implies for additive attention at
       initialization.

1. [extended] **A learned alignment.** ●
   @Bahdanau.Cho.Bengio.2014 observed that attention weights trained
   only for translation behave like alignments (the figure).
   The same effect appears in a toy model without recurrent networks. Draw
   source sequences of $8$ tokens uniformly from a vocabulary of $16$, and
   let the target at output position $j$ be the source token at position
   $\pi(j)$ for a fixed permutation $\pi$ of the positions. Score one
   learned query per output position against keys that a learned linear map
   computes from the concatenated one-hot token and one-hot position of each
   source token, use the one-hot source tokens as values, and train on the
   negative logarithm of the output entry that belongs to the target token.
    1. Implement the model with `DotProductAttention`, train it, and plot
       the attention weights of one sequence with `d2l.show_heatmaps`.
       Compare the map with $\pi$.
    1. Replace the score by the additive rule the equation and
       compare the number of steps that each model needs to reach $99\%$
       accuracy.
    1. Give queries and keys the width $d \in \{16, 256, 1024\}$ with
       standard normal initialization, and train with and without the
       factor $1/\sqrt{d}$. Relate the training curves to the entropy of the
       weights at initialization.
    1. Explain why a fixed query per output position cannot learn a
       reordering that depends on the tokens, such as swapping two
       neighbors only when the first belongs to a designated subset of the
       vocabulary, and state what the query would have to depend on.

[Discussions](https://d2l.discourse.group/t/36769)